# Experiment 2: how EQ and reverb change descriptor similarity

This example uses saved model scores. No model weights or audio are loaded unless
you enable the optional final cell. Install `pip install -e ".[notebooks]"` first.

For each source, effect and descriptor, we have five strengths: 0.2–1.0.
The question is whether similarity to the target word increases with strength.


In [ ]:
from pathlib import Path
import pandas as pd

# Run Jupyter from the repository root or from notebooks/.
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "pyproject.toml").exists():
    raise FileNotFoundError("Open this notebook from the repository root or notebooks/.")


## 1. Read one trajectory

Example: LAION-CLAP, guitar, EQ, bright. Each row is one effect strength.


In [ ]:
scores = pd.read_csv(ROOT / "results/reference/experiment2/laion-clap.csv")
trajectory = scores[
    (scores.source_type == "guitar")
    & (scores.effect_type == "eq")
    & (scores.descriptor == "bright")
].sort_values("scale")
display(trajectory[["scale", "orig_target_sim", "manip_target_sim", "delta_target_sim"]])


## 2. Calculate the metrics for this trajectory

Delta compares processed audio with the **original source**, using the same target
text. Slope and correlations compare delta with effect strength. Final delta is
the value at strength 1.0. A positive slope need not imply a positive final delta.


In [ ]:
import numpy as np
from scipy.stats import pearsonr, spearmanr

strength = trajectory["scale"].to_numpy()
delta = (trajectory["manip_target_sim"] - trajectory["orig_target_sim"]).to_numpy()

slope = np.polyfit(strength, delta, 1)[0]
print("Slope:", slope)
print("Final delta:", delta[-1])
print("Pearson r:", pearsonr(strength, delta).statistic)
print("Spearman rho:", spearmanr(strength, delta).statistic)


## 3. Summarize all four models with the shared analysis code

Each model has 120 trajectories. Positive-slope and positive-final-delta rates
average their boolean outcomes. All-source consistency requires all three sources
to have positive slopes for a descriptor/effect pair. See `analysis.py` for details.


In [ ]:
from timbre_semantics.design import load_design
from timbre_semantics.analysis import analyze_experiment2

design = load_design(ROOT / "configs/experiment2.json")
summary = analyze_experiment2(
    ROOT / "results/reference/experiment2", ROOT / "outputs/analysis", design
)
display(summary)


## 4. Optional: generate new model scores from real audio

Read `run_experiment2()` in `experiments.py` for the encoding and baseline-matching
workflow. Install the selected model extra and supply the actual recordings first.
A filename inventory alone does not authenticate audio as the paper stimuli.


In [ ]:
RUN_INFERENCE = False
MODEL = "laion-clap"
DATA_ROOT = ROOT / "data/experiment2"
OUTPUT = ROOT / "outputs/experiment2" / MODEL

if RUN_INFERENCE:
    from timbre_semantics.experiments import run_experiment2

    run_experiment2(MODEL, DATA_ROOT, OUTPUT, design)
